# Klasifikasi Biner: Sentimen Ulasan Film (IMDB)

**Buku:** *Deep Learning dengan Python* — Bab 3, Memulai dengan Jaringan Saraf

Mengklasifikasikan ulasan film sebagai positif atau negatif berdasarkan teksnya, memakai representasi *multi-hot* dan tumpukan lapisan `Dense`.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Kode bersifat *backend-agnostik*: dapat dijalankan dengan backend TensorFlow, JAX, atau PyTorch.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- `keras.Input` menggantikan `input_shape=`; `RMSprop(lr=...)` → `RMSprop(learning_rate=...)`.
- Kunci riwayat `acc`/`val_acc` → `accuracy`/`val_accuracy`.
- Perbaikan bug grafik asli: label sumbu-y grafik akurasi tertulis *Loss*.
- Fungsi `vectorize_sequences` dipakai ulang; ditambah fungsi `bangun_model()` agar pelatihan ulang bersih.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. Memuat dataset IMDB
Hanya 10.000 kata paling sering yang dipertahankan.

In [ ]:
from keras.datasets import imdb

(data_latih, label_latih), (data_uji, label_uji) = imdb.load_data(num_words=10000)
print(len(data_latih), "ulasan latih |", len(data_uji), "ulasan uji")
print("Indeks kata maksimum:", max(max(s) for s in data_latih))

Mengembalikan satu ulasan ke teks (indeks digeser 3 karena 0, 1, 2 dicadangkan untuk *padding*, awal sekuens, dan kata tak dikenal):

In [ ]:
indeks_kata = imdb.get_word_index()
indeks_balik = {nilai: kata for kata, nilai in indeks_kata.items()}
ulasan = " ".join(indeks_balik.get(i - 3, "?") for i in data_latih[0])
print(ulasan[:400], "...")

## 2. Vektorisasi multi-hot

In [ ]:
import numpy as np

def vektorisasi(sekuens, dimensi=10000):
    hasil = np.zeros((len(sekuens), dimensi), dtype="float32")
    for i, s in enumerate(sekuens):
        hasil[i, s] = 1.0
    return hasil

x_latih = vektorisasi(data_latih)
x_uji = vektorisasi(data_uji)
y_latih = np.asarray(label_latih).astype("float32")
y_uji = np.asarray(label_uji).astype("float32")
print(x_latih.shape, x_latih[0][:10])

## 3. Model

In [ ]:
from keras import layers

def bangun_model():
    model = keras.Sequential([
        keras.Input(shape=(10000,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=1e-3),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

model = bangun_model()
model.summary()

## 4. Validasi: sisihkan 10.000 sampel

In [ ]:
x_val, x_latih_sebagian = x_latih[:10000], x_latih[10000:]
y_val, y_latih_sebagian = y_latih[:10000], y_latih[10000:]

riwayat = model.fit(x_latih_sebagian, y_latih_sebagian,
                    epochs=20, batch_size=512,
                    validation_data=(x_val, y_val))

In [ ]:
import matplotlib.pyplot as plt

def plot_riwayat(riwayat, metrik="accuracy"):
    h = riwayat.history
    epoch = range(1, len(h["loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(epoch, h["loss"], "o", label="Loss latih")
    ax[0].plot(epoch, h["val_loss"], "-", label="Loss validasi")
    ax[0].set(xlabel="Epoch", ylabel="Loss", title="Loss latih dan validasi")
    ax[1].plot(epoch, h[metrik], "o", label="Latih")
    ax[1].plot(epoch, h["val_" + metrik], "-", label="Validasi")
    ax[1].set(xlabel="Epoch", ylabel=metrik, title=f"{metrik} latih dan validasi")
    for a in ax: a.legend()
    plt.tight_layout(); plt.show()

plot_riwayat(riwayat)

Loss validasi mencapai minimum sekitar epoch ke-4, lalu naik: **overfitting**. Latih ulang model baru selama 4 epoch pada seluruh data latih.

In [ ]:
model = bangun_model()
model.fit(x_latih, y_latih, epochs=4, batch_size=512, verbose=0)
loss, akurasi = model.evaluate(x_uji, y_uji, verbose=0)
print(f"Akurasi uji: {akurasi:.2%}")

## 5. Prediksi data baru

In [ ]:
p = model.predict(x_uji[:5], verbose=0).ravel()
for peluang, label in zip(p, y_uji[:5]):
    print(f"P(positif) = {peluang:.3f} | label = {int(label)}")

## Coba sendiri
- Ganti jumlah lapisan tersembunyi menjadi 1 atau 3.
- Ganti jumlah unit menjadi 32 atau 64.
- Ganti `binary_crossentropy` dengan `mse`, dan `relu` dengan `tanh`.
- Ganti dataset dengan ulasan berbahasa Indonesia (mis. ulasan aplikasi).